# Plate detection — A1 (YOLO26n vs YOLOv8n)
Thin driver (OS-independent). All logic lives in `plate_detect`; this notebook only calls the CLI.

- **Local kernel** with a GPU: repo + prepared `data/` already on disk → the Setup cell just `pip install`s the package.
- **Colab kernel** (free T4): the Setup cell bootstraps everything — clones the private repo (GitHub PAT), installs the CLI, and pulls raw A1 from Kaggle (`kaggle.json`). All later cells run from the repo root, so the CLI's relative paths resolve.

In [ ]:
# === Setup ===
# LOCAL kernel: repo already on disk + data prepared -> just install the package:
#     %cd /path/to/UIT2026-DoAnCuoiKi
#     !pip install -e src/ml/plate_detection_pipeline
# COLAB kernel: run the bootstrap below (clones private repo BRANCH, installs CLI, pulls raw A1 from Kaggle).
import os, glob, shutil, getpass

IN_COLAB = "google.colab" in str(get_ipython())
REPO   = "/content/UIT2026-DoAnCuoiKi"
BRANCH = "feat/plate-detect-a1"              # package NOT merged to main yet — clone this branch
RAW    = "data/raw/kaggle_vn_plate_segment"  # layout the A1Adapter expects: {images,labels}/{train,val}

if IN_COLAB:
    # 1) clone the PRIVATE repo, feature branch (GitHub PAT with 'repo' scope; input hidden)
    if not os.path.isdir(REPO):
        tok = getpass.getpass("GitHub PAT: ")
        !git clone --branch {BRANCH} --single-branch https://{tok}@github.com/UIT-DoAnCuoiKi/UIT2026-DoAnCuoiKi.git {REPO}
    %cd {REPO}
    !git rev-parse --abbrev-ref HEAD   # confirm the feature branch is checked out

    # 2) install the package -> puts the `plate_detect` CLI on PATH
    !pip install -q -e src/ml/plate_detection_pipeline kaggle

    # 3) raw A1 data from Kaggle (needs kaggle.json: Kaggle > Settings > Create New Token)
    if not os.path.exists(os.path.expanduser("~/.config/kaggle/kaggle.json")):
        from google.colab import files
        print("Upload kaggle.json:"); files.upload()
        !mkdir -p ~/.config/kaggle && cp kaggle.json ~/.config/kaggle/ && chmod 600 ~/.config/kaggle/kaggle.json

    if not os.path.isdir(f"{RAW}/images/train"):
        !kaggle datasets download -d duydieunguyen/licenseplates -p /tmp/a1 --unzip
        # find the dir that holds images/train + labels/train, symlink RAW to it (no 2.6G copy)
        hits = glob.glob("/tmp/a1/**/images/train", recursive=True)
        assert hits, "Kaggle unzip: images/train not found — inspect /tmp/a1 and adjust"
        root = os.path.abspath(hits[0][: -len("/images/train")])
        os.makedirs(os.path.dirname(RAW), exist_ok=True)
        if os.path.islink(RAW) or os.path.exists(RAW):
            (os.unlink if os.path.islink(RAW) else shutil.rmtree)(RAW)
        os.symlink(root, os.path.abspath(RAW))
else:
    # local kernel: assume cwd is the repo root and data/ already present
    !pip install -q -e src/ml/plate_detection_pipeline

# 4) sanity-check the raw layout the adapter reads (train + val, images + labels)
for s in ("train", "val"):
    for k in ("images", "labels"):
        assert os.path.isdir(f"{RAW}/{k}/{s}"), f"missing {RAW}/{k}/{s} — check Kaggle split names (val vs valid)"
print("OK — CLI installed, raw A1 ready at", RAW)

In [2]:
import torch; print('CUDA:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

CUDA: True Tesla T4


## 1. Prepare (class-map gate → split → dedup train↔test & train↔val → validate)

In [3]:
!plate_detect prepare

/bin/bash: line 1: plate_detect: command not found


In [4]:
!plate_detect check

/bin/bash: line 1: plate_detect: command not found


## 2. Train — full matrix @640 (both models × seeds 0,1,2)

In [5]:
!plate_detect train --imgsz 640 --seeds 0,1,2 --project runs

/bin/bash: line 1: plate_detect: command not found


## 3. imgsz ablation @960 (single seed, both models)

In [6]:
!plate_detect train --imgsz 960 --seeds 0 --project runs

/bin/bash: line 1: plate_detect: command not found


## 4. Export best → ONNX (per model & imgsz), parity-checked

In [7]:
# example; repeat per model/imgsz best run:
!plate_detect export --weights runs/yolo26n_s0_640/weights/best.pt --out weights/yolo26n_a1_640.onnx --imgsz 640

/bin/bash: line 1: plate_detect: command not found


## 5. Evaluate on A1 test → comparison table + experiments.csv

In [8]:
!plate_detect eval --imgszs 640,960 --project runs --weights-dir weights --sample-image data/processed/a1_det/images/test/$(ls data/processed/a1_det/images/test | head -1)

ls: cannot access 'data/processed/a1_det/images/test': No such file or directory
/bin/bash: line 1: plate_detect: command not found
